# Startup Outcome Prediction — End-to-End Machine Learning Pipeline

This notebook implements a complete machine learning pipeline to predict whether a startup is likely to succeed,
defined as being acquired or achieving an IPO, based on its funding history and company profile data.

The pipeline follows 21 structured steps covering data preparation, model development, evaluation, and optimisation.

## Step 1: Import Libraries

All necessary libraries are imported at the outset. `scikit-learn` provides the modelling and evaluation tools,
`pandas` and `numpy` handle data manipulation, and `warnings` suppresses non-critical alerts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Libraries imported successfully.')

## Step 2: Load Dataset

The dataset is loaded directly from the CSV file. The `low_memory=False` argument prevents mixed-type inference warnings for columns with inconsistent values.

In [ ]:
# If running in Google Colab, upload the CSV file first or mount Google Drive
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('big_startup_secsees_dataset.csv', low_memory=False)

print(f'Dataset loaded successfully.')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Step 3: Inspect Dataset Structure

A brief structural inspection reveals the data types, the presence of mixed-type columns, and an overview
of the target variable distribution prior to any filtering.

In [ ]:
print('--- Data Types ---')
print(df.dtypes)

print('\n--- First 5 Rows ---')
display(df.head())

print('\n--- Status Value Counts ---')
print(df['status'].value_counts())

## Step 4: Filter Target Classes and Define Binary Target

Only companies with a confirmed final outcome are retained. The `operating` class is excluded because these
companies have not yet reached a terminal state — their inclusion would introduce label ambiguity and degrade
model reliability. A binary target is created:

- **1 — Successful**: status is `acquired` or `ipo`
- **0 — Unsuccessful**: status is `closed`

In [ ]:
df = df[df['status'].isin(['acquired', 'ipo', 'closed'])].copy()
df['target'] = (df['status'].isin(['acquired', 'ipo'])).astype(int)

print(f'Filtered dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['target'].value_counts())
print(f'  1 = Successful (acquired or IPO): {(df["target"]==1).sum()}')
print(f'  0 = Unsuccessful (closed):        {(df["target"]==0).sum()}')

## Step 5: Clean Column Names

Column names are standardised to lowercase with leading and trailing whitespace removed, ensuring consistent
programmatic access throughout the pipeline.

In [ ]:
df.columns = df.columns.str.strip().str.lower()
print('Column names standardised:')
print(list(df.columns))

## Step 6: Check Missing Values and Duplicates

Missing value counts are inspected for each column. Geographic fields such as `state_code`, `region`,
and `city` show high missingness, which is addressed during preprocessing via imputation.
Duplicate rows are identified and removed.

In [ ]:
print('--- Missing Values ---')
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_summary = pd.DataFrame({'Missing': missing, 'Percentage (%)': missing_pct})
display(missing_summary[missing_summary['Missing'] > 0])

duplicates = df.duplicated().sum()
print(f'\nDuplicate rows found: {duplicates}')
df = df.drop_duplicates()
print(f'Dataset shape after deduplication: {df.shape}')

## Step 7: Clean Funding and Date Fields

The `funding_total_usd` column contains non-numeric characters such as commas and hyphens.
These are removed before converting to a numeric type using `pd.to_numeric` with `errors='coerce'`,
which converts unparseable values to `NaN` for later imputation. Date columns are parsed using
`pd.to_datetime` with error coercion.

In [ ]:
# Clean funding column
df['funding_total_usd'] = pd.to_numeric(
    df['funding_total_usd'].astype(str).str.replace(',', '').str.strip(),
    errors='coerce'
)

# Parse date columns
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Funding column dtype:', df['funding_total_usd'].dtype)
print('Funding column sample stats:')
print(df['funding_total_usd'].describe())

print('\nDate columns dtypes:')
for col in ['founded_at', 'first_funding_at', 'last_funding_at']:
    print(f'  {col}: {df[col].dtype}')

## Step 8: Engineer Date-Based Features

Several derived temporal features are constructed from the date columns. Year-level features
(`founded_year`, `first_funding_year`, `last_funding_year`) capture the time period of activity.
Duration features (`time_to_first_funding_days`, `time_between_first_and_last_funding_days`)
measure the pace of funding progression, which may be informative of investor confidence.

The primary business sector is extracted as the first entry from the pipe-delimited `category_list` field.

In [ ]:
df['founded_year']       = df['founded_at'].dt.year
df['first_funding_year'] = df['first_funding_at'].dt.year
df['last_funding_year']  = df['last_funding_at'].dt.year

df['time_to_first_funding_days'] = (
    df['first_funding_at'] - df['founded_at']
).dt.days

df['time_between_first_and_last_funding_days'] = (
    df['last_funding_at'] - df['first_funding_at']
).dt.days

# Extract primary business sector
df['primary_category'] = (
    df['category_list']
    .fillna('Unknown')
    .str.split('|')
    .str[0]
    .str.strip()
)

print('Engineered features created.')
display(df[['founded_year', 'first_funding_year', 'last_funding_year',
            'time_to_first_funding_days', 'time_between_first_and_last_funding_days',
            'primary_category']].describe(include='all'))

## Step 9: Exploratory Data Analysis

Exploratory analysis examines the distribution of the target variable, the distribution of key
numerical features, and the relationship between funding amounts and startup outcomes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Target distribution
target_counts = df['target'].value_counts()
axes[0].bar(['Unsuccessful (0)', 'Successful (1)'], target_counts.values, color=['#dc2626', '#16a34a'])
axes[0].set_title('Target Distribution')
axes[0].set_ylabel('Count')

# Funding total by target
df.boxplot(column='funding_total_usd', by='target', ax=axes[1],
           boxprops=dict(color='#374151'), medianprops=dict(color='#1d4ed8'))
axes[1].set_title('Total Funding by Outcome')
axes[1].set_xlabel('Target (0=Unsuccessful, 1=Successful)')
axes[1].set_ylabel('Funding (USD)')
axes[1].set_yscale('log')
plt.sca(axes[1])
plt.title('Total Funding by Outcome')

# Funding rounds by target
df.groupby('target')['funding_rounds'].mean().plot(
    kind='bar', ax=axes[2], color=['#dc2626', '#16a34a']
)
axes[2].set_title('Average Funding Rounds by Outcome')
axes[2].set_xlabel('Target (0=Unsuccessful, 1=Successful)')
axes[2].set_ylabel('Mean Funding Rounds')
axes[2].set_xticklabels(['Unsuccessful', 'Successful'], rotation=0)

plt.tight_layout()
plt.show()

# Top sectors
print('\n--- Top 10 Business Sectors ---')
print(df['primary_category'].value_counts().head(10))

# Success rate by sector (top 10)
print('\n--- Success Rate by Sector (Top 10 by volume) ---')
top_sectors = df['primary_category'].value_counts().head(10).index
sector_success = (
    df[df['primary_category'].isin(top_sectors)]
    .groupby('primary_category')['target']
    .mean()
    .sort_values(ascending=False)
    .mul(100)
    .round(1)
)
print(sector_success.to_string())

## Step 10: Select Features and Target

Features are selected based on the project specification. Identifiers (`permalink`, `name`, `homepage_url`)
and the original status column are excluded to prevent target leakage.

In [ ]:
FEATURE_COLS = [
    'primary_category',
    'funding_total_usd',
    'country_code',
    'state_code',
    'region',
    'city',
    'funding_rounds',
    'founded_year',
    'first_funding_year',
    'last_funding_year',
    'time_to_first_funding_days',
    'time_between_first_and_last_funding_days',
]

TARGET_COL = 'target'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape:  {y.shape}')
print(f'\nFeatures selected: {FEATURE_COLS}')

## Step 11: Define Categorical and Numerical Columns

Features are separated into categorical and numerical groups to enable appropriate preprocessing
strategies for each type within the `ColumnTransformer`.

In [ ]:
CATEGORICAL_COLS = [
    'primary_category',
    'country_code',
    'state_code',
    'region',
    'city',
]

NUMERICAL_COLS = [
    'funding_total_usd',
    'funding_rounds',
    'founded_year',
    'first_funding_year',
    'last_funding_year',
    'time_to_first_funding_days',
    'time_between_first_and_last_funding_days',
]

print(f'Categorical features ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}')
print(f'Numerical features  ({len(NUMERICAL_COLS)}): {NUMERICAL_COLS}')

## Step 12: Train-Test Split

A stratified 80/20 split is applied to ensure that the class proportions in the training and
test sets mirror those in the full dataset. Stratification is particularly important here given
the mild class imbalance between successful and unsuccessful outcomes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size:     {X_test.shape[0]}')
print(f'\nTraining target balance:')
print(y_train.value_counts(normalize=True).round(3))
print(f'\nTest target balance:')
print(y_test.value_counts(normalize=True).round(3))

## Step 13: Build Preprocessing Pipeline Using ColumnTransformer

A `ColumnTransformer` applies different preprocessing steps to each feature group:

- **Numerical**: median imputation followed by standard scaling.
- **Categorical**: constant-value imputation ('Unknown') followed by one-hot encoding.

Using `handle_unknown='ignore'` in the `OneHotEncoder` ensures that unseen categories at prediction
time do not cause errors.

In [ ]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, NUMERICAL_COLS),
    ('cat', categorical_transformer, CATEGORICAL_COLS),
])

print('Preprocessing pipeline constructed.')
print('  Numerical transformer: median imputer → standard scaler')
print('  Categorical transformer: constant imputer → one-hot encoder')

## Step 14: Train Logistic Regression

Logistic Regression is used as the interpretable baseline model. It assumes a linear relationship
between features and the log-odds of the outcome, making it fast and easy to interpret.
`class_weight='balanced'` adjusts for the class imbalance present in the dataset.

In [ ]:
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')),
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print('Logistic Regression trained.')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_lr, target_names=['Unsuccessful', 'Successful']))

## Step 15: Train Decision Tree

The Decision Tree provides high interpretability through its rule-based structure. However, without
depth constraints it is prone to overfitting. A `max_depth` of 10 is applied to balance complexity
against generalisation.

In [ ]:
dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced')),
])

dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

print('Decision Tree trained.')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_dt, target_names=['Unsuccessful', 'Successful']))

## Step 16: Train Random Forest

Random Forest is an ensemble of decision trees trained on bootstrapped subsets of the data.
It reduces overfitting through averaging and introduces feature randomisation at each split.
It is selected as the primary model candidate.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
        n_estimators=100, random_state=42,
        class_weight='balanced', n_jobs=-1
    )),
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print('Random Forest trained.')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_rf, target_names=['Unsuccessful', 'Successful']))

## Step 17: Evaluate All Models

Each model is evaluated using accuracy, precision, recall, F1-score, and confusion matrices.
Three-fold stratified cross-validation is applied to the training set to assess the stability of
each model's performance across different data partitions.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score':  round(f1_score(y_true, y_pred, zero_division=0), 4),
    }

results = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, y_pred_lr),
    evaluate_model('Decision Tree',       y_test, y_pred_dt),
    evaluate_model('Random Forest',       y_test, y_pred_rf),
])

print('--- Model Comparison ---')
display(results)

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, y_pred) in zip(axes, [
    ('Logistic Regression', y_pred_lr),
    ('Decision Tree',       y_pred_dt),
    ('Random Forest',       y_pred_rf),
]):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Unsuccessful', 'Successful'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

# Cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print('\n--- 3-Fold Cross-Validation (F1-Score) ---')
for name, pipeline in [
    ('Logistic Regression', lr_pipeline),
    ('Decision Tree',       dt_pipeline),
    ('Random Forest',       rf_pipeline),
]:
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f'{name}: {scores.round(4)} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}')

## Step 18: Hyperparameter Tuning for Random Forest

`GridSearchCV` is applied exclusively to the Random Forest, as it is the leading model candidate.
The parameter grid is kept compact to balance exploration with computational efficiency:

- `n_estimators`: [100, 150] — controls the number of trees
- `max_depth`: [10, 12] — limits tree depth to reduce overfitting
- `min_samples_split`: [10] — minimum samples required to split an internal node
- `min_samples_leaf`: [5] — minimum samples required at a leaf node

Three-fold cross-validation is used within the search.

In [ ]:
param_grid = {
    'classifier__n_estimators':      [100, 150],
    'classifier__max_depth':         [10, 12],
    'classifier__min_samples_split': [10],
    'classifier__min_samples_leaf':  [5],
}

rf_tuned_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, class_weight='balanced')),
])

grid_search = GridSearchCV(
    rf_tuned_pipeline,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1-Score: {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print('\n--- Tuned Random Forest Classification Report ---')
print(classification_report(y_test, y_pred_best, target_names=['Unsuccessful', 'Successful']))

## Step 19: Compare Models

A final comparison table is produced, including all three base models and the tuned Random Forest,
to facilitate an informed model selection decision.

In [ ]:
final_results = pd.DataFrame([
    evaluate_model('Logistic Regression',     y_test, y_pred_lr),
    evaluate_model('Decision Tree',           y_test, y_pred_dt),
    evaluate_model('Random Forest',           y_test, y_pred_rf),
    evaluate_model('Random Forest (Tuned)',   y_test, y_pred_best),
])

print('--- Final Model Comparison ---')
display(final_results)

# Bar chart comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 5))
colours = ['#6b7280', '#f59e0b', '#3b82f6', '#16a34a']

for i, (_, row) in enumerate(final_results.iterrows()):
    ax.bar(x + i * width, row[metrics].values, width, label=row['Model'], color=colours[i])

ax.set_xlabel('Metric')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

## Step 20: Select Final Model and Explain Feature Importance

The tuned Random Forest is selected as the final model. Feature importances derived from the
fitted classifier indicate the relative contribution of each feature to the model's decisions.
The top 15 features are visualised below.

In [ ]:
rf_classifier = best_model.named_steps['classifier']

ohe_feature_names = (
    best_model.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(CATEGORICAL_COLS)
    .tolist()
)

all_feature_names = NUMERICAL_COLS + ohe_feature_names
importances = pd.Series(rf_classifier.feature_importances_, index=all_feature_names)
top_features = importances.sort_values(ascending=False).head(15)

print('Top 15 Feature Importances:')
print(top_features.round(4).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', ax=ax, color='#3b82f6')
ax.set_title('Top 15 Feature Importances — Tuned Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

## Step 21: Save Final Conclusions

The modelling process is summarised with key findings, model selection rationale, and observations
regarding feature importance.

In [ ]:
conclusions = """
=============================================================
FINAL CONCLUSIONS
=============================================================

Three classification models were trained and evaluated on a binary
prediction task derived from the startup funding dataset.

Logistic Regression served as an interpretable baseline and achieved
competitive performance for a linear model, demonstrating that funding
and temporal features contain genuinely predictive signal.

Decision Tree provided strong interpretability but showed greater
susceptibility to overfitting than the ensemble approaches.

Random Forest consistently achieved the highest scores across all
evaluation metrics. Hyperparameter tuning via GridSearchCV further
refined performance, particularly in recall and F1-score.

The most influential features were funding-related variables
(total funding and number of rounds) and temporal features
(founding year, time to first funding, duration of funding activity).

SELECTED MODEL: Tuned Random Forest Classifier
DEPLOYMENT: Streamlit web application (app.py)

LIMITATIONS:
- Dataset is weighted towards US technology companies.
- Training data reflects historical conditions up to approximately 2015.
- Model drift may occur if applied to contemporary startup ecosystems.
- Predictions are probabilistic and should not be used as the sole
  basis for investment or strategic decisions.
=============================================================
"""

print(conclusions)